In [1]:
import nltk
from nltk.corpus import brown

nltk.download('brown')
nltk.download('universal_tagset')

# tagged sentences
sentences = brown.tagged_sents(tagset='universal')[:1000]

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.


In [2]:
from collections import defaultdict, Counter

transition = defaultdict(Counter)
emission = defaultdict(Counter)
start_prob = Counter()

for sent in sentences:
    prev_tag = None

    for i, (word, tag) in enumerate(sent):
        word = word.lower()

        if i == 0:
            start_prob[tag] += 1

        emission[tag][word] += 1

        if prev_tag:
            transition[prev_tag][tag] += 1

        prev_tag = tag

# Normalize
def normalize(counter):
    total = sum(counter.values())
    return {k: v/total for k, v in counter.items()}

start_prob = normalize(start_prob)
transition = {k: normalize(v) for k, v in transition.items()}
emission = {k: normalize(v) for k, v in emission.items()}

In [3]:
def forward(obs, states):
    alpha = [{}]

    # init
    for s in states:
        alpha[0][s] = start_prob.get(s, 0) * emission[s].get(obs[0], 1e-6)

    # recursion
    for t in range(1, len(obs)):
        alpha.append({})
        for s in states:
            alpha[t][s] = sum(
                alpha[t-1][sp] * transition[sp].get(s, 0)
                for sp in states
            ) * emission[s].get(obs[t], 1e-6)

    return sum(alpha[-1].values())

In [4]:
def viterbi(obs, states):
    V = [{}]
    path = {}

    # init
    for s in states:
        V[0][s] = start_prob.get(s, 0) * emission[s].get(obs[0], 1e-6)
        path[s] = [s]

    # recursion
    for t in range(1, len(obs)):
        new_path = {}
        V.append({})

        for s in states:
            prob, state = max(
                (V[t-1][sp] * transition[sp].get(s, 0) *
                 emission[s].get(obs[t], 1e-6), sp)
                for sp in states
            )
            V[t][s] = prob
            new_path[s] = path[state] + [s]

        path = new_path

    # best final state
    prob, state = max((V[-1][s], s) for s in states)
    return path[state]

In [8]:
def backward(obs, states):
    T = len(obs)
    beta = [{} for _ in range(T)]

    # init
    for s in states:
        beta[T-1][s] = 1

    # recursion
    for t in range(T-2, -1, -1):
        for s in states:
            beta[t][s] = sum(
                transition[s].get(sp, 0) *
                emission[sp].get(obs[t+1], 1e-6) *
                beta[t+1][sp]
                for sp in states
            )

    # final prob
    return sum(
        start_prob.get(s, 1e-6) * # Modified here to use .get()
        emission[s].get(obs[0], 1e-6) *
        beta[0][s]
        for s in states
    )

In [6]:
# Simplified placeholder (concept demonstration)

def baum_welch():
    print("Used to re-estimate transition and emission probabilities")
    print("Based on forward-backward expectations")

In [9]:
sentence = ["the", "market", "is", "good"]
states = list(emission.keys())

print("Forward Prob:", forward(sentence, states))
print("Backward Prob:", backward(sentence, states))
print("Viterbi Tags:", viterbi(sentence, states))

Forward Prob: 7.567126097896123e-14
Backward Prob: 7.567126097922096e-14
Viterbi Tags: ['DET', 'NOUN', 'VERB', 'NOUN']
